In [22]:
import rasterio
import numpy as np
import matplotlib
matplotlib.use('Agg') # Use non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import sys
from pathlib import Path

print("Starting final composite plot generation...")

Starting final composite plot generation...


In [ ]:
# --- Directories ---
BASE_DIR = Path.cwd().parent.parent
DATA_DIR = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"

# --- 1. Define File Paths and Thresholds ---
RESISTANCE_RASTER = RESULTS_DIR / "final_resistance_surface.tif"
TRAFFIC_RASTER = RESULTS_DIR / "final_corridor_traffic.tif"
# This is the map of costs *only* on the corridors
BOTTLENECK_RASTER = RESULTS_DIR / "corridor_bottlenecks.tif"
OUTPUT_PLOT = BASE_DIR / "final_annotated_composite_map.png"

# --- Analysis Settings ---
# Set the cost value above which we draw a circle
# Since your barrier is 10000, 2000 is a good start
BOTTLENECK_COST_THRESHOLD = 2000.0 
# Set the percentile of traffic to show (e.g., top 50%)
TRAFFIC_PERCENTILE_THRESHOLD = 50.0
# Your barrier cost and the NoData value from previous scripts
EXTREME_BARRIER_COST = 10000.0
NODATA_VALUE = -9999.0

In [30]:
# --- 2. Load Base Resistance Raster (for plotting) ---
print(f"Loading base resistance raster: {RESISTANCE_RASTER}")
with rasterio.open(RESISTANCE_RASTER) as src:
    meta = src.meta.copy()
    resistance_array = src.read(1).astype(np.float32)
    # Apply the same robust cleaning as the worker script
    resistance_array = np.nan_to_num(
        resistance_array, 
        nan=EXTREME_BARRIER_COST,
        posinf=EXTREME_BARRIER_COST,
        neginf=1.0
    )
    if meta['nodata'] is not None:
        resistance_array[resistance_array == meta['nodata']] = EXTREME_BARRIER_COST
    resistance_array[resistance_array <= 0] = 1.0
    
    # For plotting, set barriers to 'nan' so they are white
    resistance_plot = resistance_array.copy().astype(float)
    resistance_plot[resistance_plot == EXTREME_BARRIER_COST] = np.nan

Loading base resistance raster: c:\ZHAW\5.Semester\PA2\PA2-Modelling_Wildlife_Corridors\results\final_resistance_surface.tif


In [31]:
# --- 3. Load Traffic Raster (for corridor overlay) ---
print(f"Loading traffic raster: {TRAFFIC_RASTER}")
with rasterio.open(TRAFFIC_RASTER) as src:
    traffic_array = src.read(1).astype(np.float32)
    
    # Find the threshold value (e.g., P90)
    non_zero_pixels = traffic_array[traffic_array > 0]
    if non_zero_pixels.size == 0:
        print("Error: No traffic data found.")
        sys.exit(1)
        
    traffic_threshold_val = np.percentile(non_zero_pixels, TRAFFIC_PERCENTILE_THRESHOLD)
    print(f"Traffic threshold (P{TRAFFIC_PERCENTILE_THRESHOLD}) is: {traffic_threshold_val} crossings")
    
    # Create a masked array to hide all non-corridor pixels
    traffic_masked = np.ma.masked_less(traffic_array, traffic_threshold_val)

Loading traffic raster: c:\ZHAW\5.Semester\PA2\PA2-Modelling_Wildlife_Corridors\results\final_corridor_traffic.tif
Traffic threshold (P50.0) is: 42.0 crossings


In [32]:
# --- 4. Create the Bottleneck Map ---
# Create a mask where traffic is BELOW our threshold
# These are the pixels we want to hide
mask = traffic_array < traffic_threshold_val

# Create our new output array by copying the resistance values
corridor_bottlenecks = resistance_array.copy()

# Apply the mask: where traffic is low, set to NoData
corridor_bottlenecks[mask] = NODATA_VALUE
print("Bottleneck mask applied.")

# --- 5. Save the New Raster ---
# Update the metadata for our new output file
meta.update({
    'dtype': 'float32',
    'nodata': NODATA_VALUE
})

print(f"Saving bottleneck map to {BOTTLENECK_RASTER}...")
with rasterio.open(BOTTLENECK_RASTER, 'w', **meta) as dst:
    dst.write(corridor_bottlenecks.astype(np.float32), 1)

print("Analysis complete.")

Bottleneck mask applied.
Saving bottleneck map to c:\ZHAW\5.Semester\PA2\PA2-Modelling_Wildlife_Corridors\results\corridor_bottlenecks.tif...
Analysis complete.


In [33]:
# --- 5. Create the Composite Plot ---
print("Generating composite plot...")
fig, ax = plt.subplots(figsize=(15, 15))

# --- Layer 1: Base Resistance ---
cmap_base = plt.colormaps.get('Greys').copy()
cmap_base.set_bad(color='white') # Show 'nan' (barriers) as white
vmax_resistance = np.nanpercentile(resistance_plot, 99) # Get a good max value
im_base = ax.imshow(
    resistance_plot, 
    cmap=cmap_base, 
    norm=colors.LogNorm(vmin=1, vmax=vmax_resistance)
)
# Add a colorbar for the base map
cbar_base = fig.colorbar(im_base, ax=ax, shrink=0.5, pad=0.02, label='Resistance Cost (Log Scale)')

# --- Layer 2: Corridor Overlay ---
cmap_traffic = plt.colormaps.get('viridis').copy()
cmap_traffic.set_bad(color='none') # Make non-corridor pixels transparent
im_traffic = ax.imshow(
    traffic_masked, 
    cmap=cmap_traffic,
    norm=colors.LogNorm(vmin=traffic_threshold_val, vmax=traffic_array.max()),
    alpha=0.7 # Make it semi-transparent
)
# Add a colorbar for the traffic
cbar_traffic = fig.colorbar(im_traffic, ax=ax, shrink=0.5, pad=0.1, label='LCP Traffic (Top 10%)')

# --- Layer 3: Bottleneck Circles ---
ax.scatter(
    cols, 
    rows, 
    s=80,             # Size of the circles
    facecolors='none',# Hollow circles
    edgecolors='red', # Red outline
    linewidths=0.8,   # Thin outline
    label=f'Bottleneck (Cost > {BOTTLENECK_COST_THRESHOLD})'
)

Generating composite plot...


In [34]:
# --- 6. Finalize and Save ---
ax.set_title("Final Composite Map: Corridors and Bottlenecks", fontsize=20)
ax.set_xlabel('Easting (Pixel Coordinates)')
ax.set_ylabel('Northing (Pixel Coordinates)')
ax.legend(loc='upper right', facecolor='white', framealpha=0.9, markerscale=1.5)
plt.tight_layout()

print(f"Saving final plot to {OUTPUT_PLOT}...")
plt.savefig(OUTPUT_PLOT, dpi=150, bbox_inches="tight")

print("Done.")

Saving final plot to c:\ZHAW\5.Semester\PA2\PA2-Modelling_Wildlife_Corridors\final_annotated_composite_map.png...
Done.
